In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle

In [10]:
data = pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [11]:
data = data.drop(['RowNumber','CustomerId','Surname'], axis=1)
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [12]:
# geography and gender are the catagorical features.
#encoding the catagorical features
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1


In [13]:
#now for gender column fit transform was easy as for 2 values but for geography column we have 3 values so label endcoder 
# it will give france =0 , germany = 1 and spain=2 but that would mean france>germany>spain

#use one hot encoding 
from sklearn.preprocessing import OneHotEncoder
one_hot_encoder_geo = OneHotEncoder()
geo_encoded = one_hot_encoder_geo.fit_transform(data[['Geography']])
geo_encoded


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [14]:
one_hot_encoder_geo.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [15]:
df_geo_encoded = pd.DataFrame(geo_encoded.toarray(),columns=one_hot_encoder_geo.get_feature_names_out(['Geography']))
df_geo_encoded
#toarray is important as it will convert the sparse matrix to a dense matrix which can be converted to a dataframe

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [16]:
#combine with the original data
data = pd.concat([data.drop('Geography', axis=1), df_geo_encoded], axis=1)
data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


In [17]:
#save the label encoder and the one hot encoder and the final file
with open('label_encode_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)
with open('one_hot_encoder_geo.pkl','wb') as file:
    pickle.dump(one_hot_encoder_geo,file)




In [18]:
#divide the dataset into dependent and independent features
X = data.drop(['Exited'], axis=1)
y = data['Exited']

#split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=42)

#feature scaling (don't use them in tree based models)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)



In [19]:
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

ANN implimentation

In [20]:
X_train.shape[0]

8000

In [1]:
# import tensorflow as tf
import keras
from keras.models import Sequential
from keras.layers import Dense
from keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [22]:
#the 1st hidden layer will get the input shape which is the input layer and we don't need it in further steps
#(,) - in the input shape as it gets a tuple with single dimension

model = Sequential([
    Dense(64,activation = 'relu', input_shape=(X_train.shape[1],)),# HL1 connected with input layer
    Dense(32,activation = 'relu'),# HL2 connected with HL1
    Dense(1,activation = 'sigmoid') #output layer connected with HL2
])



In [23]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [ ]:
#one way of model compiling

# import tensorflow
# opt = tensorflow.keras.optimizers.Adam(learning_rate=0.01)
# loss = tensorflow.keras.losses.BinaryCrossentropy()

import keras
opt = keras.optimizers.Adam(learning_rate=0.01)
loss = keras.losses.BinaryCrossentropy()

#one type is gradient descent that we already know
#next is adam- adaptive moment estimation - [adagrad + rmsprop]
#uses moving average of gradient and adjusts LR per parameter
loss

In [25]:
#Before applying forward and backward propagation we need to compile the model and specify the loss function and the optimizer and the metrics we want to track
#2nd way
model.compile(optimizer=opt,loss = 'binary_crossentropy', metrics=['accuracy'])

In [2]:
#set up the tesnorboard 
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir=log_dir,histogram_freq=1)


In [27]:
#let say we have 100 epoch and while training we want to stop the training if the validation loss is not decreasing for 5 consecutive epochs then we can use early stopping callback

early_stopping_callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
#patient = 5 means be patient for 5 epochs and if no imporvement then stop, restore_best_weights means get me the best weights out before the ealry stopping
 

In [28]:
#training the model
history = model.fit(
    X_train, y_train, validation_data = (X_test, y_test),
    epochs = 100,
    callbacks = [tensorflow_callback, early_stopping_callback]
)

Epoch 1/100
250/250 [==============================] - 0s 672us/step - loss: 0.3941 - accuracy: 0.8331 - val_loss: 0.3501 - val_accuracy: 0.8540
Epoch 2/100
250/250 [==============================] - 0s 382us/step - loss: 0.3532 - accuracy: 0.8550 - val_loss: 0.3847 - val_accuracy: 0.8500
Epoch 3/100
250/250 [==============================] - 0s 378us/step - loss: 0.3521 - accuracy: 0.8586 - val_loss: 0.3438 - val_accuracy: 0.8525
Epoch 4/100
250/250 [==============================] - 0s 376us/step - loss: 0.3429 - accuracy: 0.8619 - val_loss: 0.3396 - val_accuracy: 0.8615
Epoch 5/100
250/250 [==============================] - 0s 375us/step - loss: 0.3383 - accuracy: 0.8634 - val_loss: 0.3389 - val_accuracy: 0.8600
Epoch 6/100
250/250 [==============================] - 0s 374us/step - loss: 0.3373 - accuracy: 0.8622 - val_loss: 0.3401 - val_accuracy: 0.8545
Epoch 7/100
250/250 [==============================] - 0s 373us/step - loss: 0.3359 - accuracy: 0.8637 - val_loss: 0.3519 - val_ac

In [29]:
#save model file and h5 file is compatible with keras
model.save('ann_model.h5')

/Users/kshitoki/Documents/annclassification/venv/lib/python3.11/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [30]:
#load tensorboard extension
%load_ext tensorboard


The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [37]:
%tensorboard --logdir logs/fit/

Reusing TensorBoard on port 6010 (pid 63119), started 0:00:07 ago. (Use '!kill 63119' to kill it.)

In [32]:
import pkg_resources
print("pkg_resources working")

pkg_resources working


In [33]:
!{sys.executable} -m pip uninstall -y setuptools
!{sys.executable} -m pip install setuptools==68.0.0

zsh:1: parse error near `-m'
zsh:1: parse error near `-m'


In [34]:
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [ ]:
%tensorboard --logdir logs/fit/

Reusing TensorBoard on port 6009 (pid 62988), started 0:01:26 ago. (Use '!kill 62988' to kill it.)

In [40]:
!kill 62988
